In [2]:
!nvidia-smi
!pip install -q ultralytics roboflow opencv-python matplotlib

Mon Nov  9 00:03:52 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 595.91.07              Driver Version: 595.91.07      CUDA Version: 13.2     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA GeForce RTX 3090        On  |   00000000:81:00.0 Off |                  N/A |
| 30%   25C    P8             31W /  350W |       1MiB /  24576MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [16]:
import os
import yaml

# Roboflow에서 다운로드한 dataset 폴더 내 data.yaml 경로 설정
DATA_YAML_PATH = "/workspace/seg_yolo-3/data.yaml"  # 본인의 데이터셋 경로로 수정

# data.yaml 내용 확인 (클래스 7개 및 경로 설정 정상 여부 체크)
if os.path.exists(DATA_YAML_PATH):
    with open(DATA_YAML_PATH, 'r') as f:
        data_config = yaml.safe_load(f)
    print("=== 데이터셋 클래스 목록 ===")
    print(data_config.get('names'))
else:
    print("⚠️ data.yaml 파일을 찾을 수 없습니다. 경로를 확인해 주세요.")

=== 데이터셋 클래스 목록 ===
['base_deck', 'block', 'hook', 'obstacle', 'robot', 'weld_joint', 'welding_spark', 'worker']


In [17]:
from ultralytics import YOLO

# 1. YOLOv11 Medium Segmentation 사전 학습 가중치 로드
model = YOLO("yolo11m-seg.pt")

# 2. RunPod RTX 3090 맞춤형 학습 시작
results = model.train(
    data=DATA_YAML_PATH,
    epochs=200,                  # 충분히 긴 학습 (Patience 설정으로 과적합 방지)
    patience=35,                 # 35 epoch 동안 mAP 개선이 없으면 조기 종료
    batch=32,                    # RTX 3090 24GB VRAM에 최적화된 배치 크기
    imgsz=(384, 640),            # (Height, Width) 16:9 비율 유지 직사각형 해상도
    rect=True,                   # 직사각형 추론 및 학습 패딩 최적화
    device=0,                    # GPU 0번 사용
    workers=8,                   # 데이터 로딩 앙상블 워커 수
    optimizer="auto",            # AdamW/SGD 자동 선택
    lr0=0.01,                    # 초기 학습률
    lrf=0.01,                    # 최종 학습률 비율
    overlap_mask=True,           # 세그멘테이션 마스크 중첩 허용
    mask_ratio=4,                # 세그멘테이션 마스크 정밀도 비율
    project="VAE_Vision",        # 결과 저장 메인 프로젝트 폴더
    name="yolo11m_seg_vae_v1",   # 세부 가중치 저장 폴더명
    save=True,                   # best.pt / last.pt 자동 저장
    exist_ok=True,               # 기존 폴더 덮어쓰기 허용
    plots=True                   # 학습 loss, mAP 그래프 자동 시각화
)

Ultralytics 8.4.208 🚀 Python-3.12.3 torch-2.8.0+cu128 CUDA:0 (NVIDIA GeForce RTX 3090, 24126MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=32, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/workspace/seg_yolo-3/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=200, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=(384, 640), iou=None, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo11m-seg.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=yolo11m_seg_vae_v1, nbs=64, nms=

In [19]:
# 학습이 완료된 최적 가중치(best.pt) 로드
best_model = YOLO("/workspace/runs/segment/VAE_Vision/yolo11m_seg_vae_v1/weights/best.pt")

# Validation 데이터셋 정밀 검증 (Box mAP 및 Segment Mask mAP 추출)
metrics = best_model.val(
    data=DATA_YAML_PATH,
    imgsz=(384, 640),
    split="val",
    rect=True
)

print("\n==========================================")
print(f"📦 Bounding Box mAP50-95 : {metrics.box.map:.4f}")
print(f"📦 Bounding Box mAP50    : {metrics.box.map50:.4f}")
print(f"🧩 Seg Mask     mAP50-95 : {metrics.seg.map:.4f}")
print(f"🧩 Seg Mask     mAP50    : {metrics.seg.map50:.4f}")
print("==========================================")

WARNING ⚠️ updating to 'imgsz=640'. 'train' and 'val' imgsz must be an integer, while 'predict' and 'export' imgsz may be a [h, w] list or an integer, i.e. 'yolo export imgsz=640,480' or 'yolo export imgsz=640'
Ultralytics 8.4.208 🚀 Python-3.12.3 torch-2.8.0+cu128 CUDA:0 (NVIDIA GeForce RTX 3090, 24126MiB)
YOLO11m-seg summary (fused): 138 layers, 22,341,480 parameters, 0 gradients, 113.0 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 561.5±88.9 MB/s, size: 42.3 KB)
val: Scanning /workspace/seg_yolo-3/valid/labels.cache... 77 images, 0 backgrounds, 1 corrupt: 100% ━━━━━━━━━━━━ 77/77 32.3Mit/s 0.0s
val: /workspace/seg_yolo-3/valid/images/SampleV1_1_mp4-36_jpg.rf.d83972105b8343c5ea7c92fd8a104a4e.jpg: ignoring corrupt image/label: labels mix segment and detection rows
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95)     Mask(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 5/5 2.9it/s 1.7s0.3ss
                   all         76        

In [14]:
# 클래스별 세부 mAP 점수 출력
for i, name in best_model.names.items():
    b_map = metrics.box.maps[i]  # Box mAP50-95
    s_map = metrics.seg.maps[i] if metrics.seg else 0  # Seg mAP50-95
    print(f"[{i}] {name:15s} | Box mAP50-95: {b_map:.4f} | Seg mAP50-95: {s_map:.4f}")

[0] base_deck       | Box mAP50-95: 0.4385 | Seg mAP50-95: 0.2641
[1] block           | Box mAP50-95: 0.5032 | Seg mAP50-95: 0.3868
[2] hook            | Box mAP50-95: 0.3742 | Seg mAP50-95: 0.1609
[3] obstacle        | Box mAP50-95: 0.3188 | Seg mAP50-95: 0.2804
[4] robot           | Box mAP50-95: 0.6320 | Seg mAP50-95: 0.4225
[5] weld_joint      | Box mAP50-95: 0.5176 | Seg mAP50-95: 0.4753
[6] welding_spark   | Box mAP50-95: 0.2632 | Seg mAP50-95: 0.2970
[7] worker          | Box mAP50-95: 0.4163 | Seg mAP50-95: 0.2050


In [13]:
# mAP50 (IOU 0.5) 기준 클래스별 정밀 지표 확인
print(f"{'Class':15s} | {'Box mAP50':10s} | {'Seg mAP50':10s}")
print("-" * 45)
for i, name in best_model.names.items():
    b_map50 = metrics.box.map50s[i]  # Box mAP50
    s_map50 = metrics.seg.map50s[i] if metrics.seg else 0  # Seg mAP50
    print(f"{name:15s} | {b_map50:.4f}     | {s_map50:.4f}")

Class           | Box mAP50  | Seg mAP50 
---------------------------------------------


AttributeError: 'Metric' object has no attribute 'map50s'. See valid attributes below.
Class for computing evaluation metrics for Ultralytics YOLO models.

    Attributes:
        p (list): Precision for each class. Shape: (nc,).
        r (list): Recall for each class. Shape: (nc,).
        f1 (list): F1 score for each class. Shape: (nc,).
        all_ap (list): AP scores for all classes and all IoU thresholds. Shape: (nc, 10).
        ap_class_index (list): Index of class for each AP score. Shape: (nc,).
        nc (int): Number of classes.
        image_metrics (dict[str, dict]): Per-image precision, recall, F1, TP, FP, and FN keyed by image name.

    Methods:
        ap50: AP at IoU threshold of 0.5 for all classes.
        ap: AP at IoU thresholds from 0.5 to 0.95 for all classes.
        mp: Mean precision of all classes.
        mr: Mean recall of all classes.
        map50: Mean AP at IoU threshold of 0.5 for all classes.
        map75: Mean AP at IoU threshold of 0.75 for all